# deutsch-deid — Quickstart

This notebook walks through **all major features** of the `deutsch-deid` library across all 11 supported entity types.

| # | Section |
|---|---|
| 1 | Installation & imports |
| 2 | Detect PII in plain text (`analyze.text`) |
| 3 | New German ID numbers — SVNR & STEUER_ID |
| 4 | Anonymize plain text — all three modes (`guard.text`) |
| 5 | Document processing — `.txt` / `.pdf` / `.docx` |
| 6 | Custom patterns |
| 7 | Config options — entity filters & score threshold |
| 8 | End-to-end German student record |

## 1. Installation & Imports

In [ ]:
!pip install deutsch-deid

In [1]:
import deutsch_deid
from deutsch_deid import analyze, guard, custom_pattern, ALL_DE_ENTITY_TYPES

print("deutsch-deid version:", deutsch_deid.__version__)
print(f"\n{len(ALL_DE_ENTITY_TYPES)} supported entity types:")
print(", ".join(sorted(ALL_DE_ENTITY_TYPES)))

deutsch-deid version: 1.0.0

11 supported entity types:
DATE, EMAIL_ADDRESS, IP_ADDRESS, LOCATION, PERSON, PHONE_NUMBER, STEUER_ID, SVNR, TIME, URL, ZIPCODE


## 2. Detect PII in Plain Text

`analyze.text()` returns a list of findings. Each finding contains:

| Key | Description |
|-----|-------------|
| `type` | Entity label (e.g. `PERSON`, `SVNR`) |
| `start` | Start character offset |
| `end` | End character offset |
| `score` | Confidence score (0.0 – 1.0) |

The sample text below is crafted to trigger **all 11 entity types**.

In [2]:
sample_text = (
    "Schülerdaten:\n"
    "Name: Anna Richter, Klasse 9A.\n"
    "Geburtsdatum: 15. April 2008.\n"
    "Adresse: Musterstraße 12, 10115 Berlin.\n"
    "Telefon: +49 162 55512345.\n"
    "E-Mail: a.richter@gymnasium-berlin.de.\n"
    "Lernportal: https://lernportal.gymnasium-berlin.de/profil.\n"
    "Unterricht beginnt täglich um 08:00 Uhr.\n"
    "Schulnetz-IP: 172.16.5.23.\n"
    "Sozialversicherungsnummer: 12 150780 J 009.\n"
    "Steuer-ID der Eltern: 12345678903.\n"
)

print(sample_text)

Schülerdaten:
Name: Anna Richter, Klasse 9A.
Geburtsdatum: 15. April 2008.
Adresse: Musterstraße 12, 10115 Berlin.
Telefon: +49 162 55512345.
E-Mail: a.richter@gymnasium-berlin.de.
Lernportal: https://lernportal.gymnasium-berlin.de/profil.
Unterricht beginnt täglich um 08:00 Uhr.
Schulnetz-IP: 172.16.5.23.
Sozialversicherungsnummer: 12 150780 J 009.
Steuer-ID der Eltern: 12345678903.



In [3]:
findings = analyze.text(sample_text)

print(f"Detected {len(findings)} PII entities:\n")
print(f"{'Type':<22} {'Matched text':<45} {'Score':>6}")
print("-" * 77)
for f in findings:
    matched = sample_text[f['start']:f['end']].replace('\n', ' ')
    print(f"{f['type']:<22} {matched:<45} {f['score']:>6.2f}")

✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Detected 12 PII entities:

Type                   Matched text                                   Score
-----------------------------------------------------------------------------
PERSON                 Anna Richter                                    0.90
DATE                   15. April 2008                                  0.85
LOCATION               Musterstraße                                    0.90
ZIPCODE                10115                                           0.75
LOCATION               Berlin                                          0.90
PHONE_NUMBER           +49 162 55512345                                0.75
EMAIL_ADDRESS          a.r

## 3. New German ID Numbers

### 3.1 Sozialversicherungsnummer (SVNR)

Structure: `[Bereich 2d][DDMMYY 6d][Buchstabe A-Z][Serien 2d][Prüfziffer 1d]`  
Example: `12 150780 J 009` — validated via SVN modular check algorithm.

**Score tiers:**
| Tier | Score |
|------|-------|
| base (regex only) | 0.40 |
| with context keyword | 0.70 |
| validated (checksum OK) | **0.90** |

In [4]:
# ── SVNR with surrounding context (e.g. in a form) ────────────────────────────
svnr_texts = [
    ("With context keyword",
     "Sozialversicherungsnummer: 12 150780 J 009"),
    ("Compact form (no spaces)",
     "SV-Nr: 12150780J009"),
    ("No context keyword",
     "12 150780 J 009"),
    ("Invalid check digit (→ score 0)",
     "Sozialversicherungsnummer: 12 150780 J 001"),
]

for label, text in svnr_texts:
    res = analyze.text(text, config={"set_entities": {"keep": ["SVNR"]}})
    if res:
        print(f"{label:<36} matched='{text[res[0]['start']:res[0]['end']]}'  score={res[0]['score']:.2f}")
    else:
        print(f"{label:<36} → not detected")

With context keyword                 matched='12 150780 J 009'  score=0.90
Compact form (no spaces)             matched='12150780J009'  score=0.90
No context keyword                   matched='12 150780 J 009'  score=0.90
Invalid check digit (→ score 0)      → not detected


### 3.2 Steuer-ID / Steuernummer / USt-IdNr (STEUER_ID)

Three sub-formats are recognised under one label:

| Sub-format | Example | Validation |
|------------|---------|------------|
| USt-IdNr (VAT, intra-community) | `DE123456788` | ISO 7064 MOD 11,10 |
| Steuernummer (business, domestic) | `12/345/67890` | none (varies by state) |
| Steuerliche Identifikationsnummer | `12345678903` | ISO 7064 MOD 11,10 |

In [5]:
tax_cases = [
    # (label, text)
    ("USt-IdNr — valid",         "USt-IdNr: DE123456788"),
    ("USt-IdNr — invalid check", "USt-IdNr: DE123456789"),
    ("Steuernummer — with slash", "Steuernummer: 12/345/67890"),
    ("Steuernummer — space form", "St.-Nr. 21 815 08155"),
    ("Individual TIN — valid",    "Steuer-ID: 12345678903"),
    ("Individual TIN — no ctx",   "12345678903"),
]

for label, text in tax_cases:
    res = analyze.text(text, config={"set_entities": {"keep": ["STEUER_ID"]}})
    if res:
        matched = text[res[0]['start']:res[0]['end']]
        print(f"{label:<36} matched='{matched}'  score={res[0]['score']:.2f}")
    else:
        print(f"{label:<36} → not detected")

USt-IdNr — valid                     matched='DE123456788'  score=0.92
USt-IdNr — invalid check             → not detected
Steuernummer — with slash            matched='12/345/67890'  score=0.85
Steuernummer — space form            matched='21 815 08155'  score=0.85
Individual TIN — valid               matched='12345678903'  score=0.92
Individual TIN — no ctx              matched='12345678903'  score=0.55


## 4. Guard Modes

`guard.text()` returns `{ "guarded_text": str, "findings": list }`.

| Mode | Behaviour |
|------|-----------|
| `anonymize` *(default)* | Realistic synthetic German values |
| `tag` | `[ENTITY_TYPE]` bracket labels |
| `i_tag` | `[ENTITY_TYPE_N]` indexed labels |

In [6]:
# ── Mode: anonymize ────────────────────────────────────────────────────────────
result_anon = guard.text(sample_text, config={"mode": "anonymize"})

print("=== ORIGINAL ===")
print(sample_text)
print("=== ANONYMIZED ===")
print(result_anon["guarded_text"])

=== ORIGINAL ===
Schülerdaten:
Name: Anna Richter, Klasse 9A.
Geburtsdatum: 15. April 2008.
Adresse: Musterstraße 12, 10115 Berlin.
Telefon: +49 162 55512345.
E-Mail: a.richter@gymnasium-berlin.de.
Lernportal: https://lernportal.gymnasium-berlin.de/profil.
Unterricht beginnt täglich um 08:00 Uhr.
Schulnetz-IP: 172.16.5.23.
Sozialversicherungsnummer: 12 150780 J 009.
Steuer-ID der Eltern: 12345678903.

=== ANONYMIZED ===
Schülerdaten:
Name: Lukas Bauer, Klasse 9A.
Geburtsdatum: 14. Januar 1983.
Adresse: Hauptstraße 12, 20095 Hamburg.
Telefon: +49 151 87654321.
E-Mail: l.bauer@beispiel.de.
Lernportal: https://www.beispiel.de/seite.
Unterricht beginnt täglich um 09:15 Uhr.
Schulnetz-IP: 10.20.30.40.
Sozialversicherungsnummer: 65 230561 M 013.
Steuer-ID der Eltern: 87654321095.



In [8]:
# ── Mode: tag ─────────────────────────────────────────────────────────────────
result_tag = guard.text(sample_text, config={"mode": "tag"})

print("=== TAG MODE ===")
print(result_tag["guarded_text"])

=== TAG MODE ===
Schülerdaten:
Name: [PERSON], Klasse 9A.
Geburtsdatum: [DATE].
Adresse: [LOCATION] 12, [ZIPCODE] [LOCATION].
Telefon: [PHONE_NUMBER].
E-Mail: [EMAIL_ADDRESS].
Lernportal: [URL].
Unterricht beginnt täglich um [TIME].
Schulnetz-IP: [IP_ADDRESS].
Sozialversicherungsnummer: [SVNR].
Steuer-ID der Eltern: [STEUER_ID].



In [9]:
# ── Mode: i_tag ───────────────────────────────────────────────────────────────
result_itag = guard.text(sample_text, config={"mode": "i_tag"})

print("=== INDEXED TAG MODE ===")
print(result_itag["guarded_text"])

=== INDEXED TAG MODE ===
Schülerdaten:
Name: [PERSON_1], Klasse 9A.
Geburtsdatum: [DATE_1].
Adresse: [LOCATION_1] 12, [ZIPCODE_1] [LOCATION_2].
Telefon: [PHONE_NUMBER_1].
E-Mail: [EMAIL_ADDRESS_1].
Lernportal: [URL_1].
Unterricht beginnt täglich um [TIME_1].
Schulnetz-IP: [IP_ADDRESS_1].
Sozialversicherungsnummer: [SVNR_1].
Steuer-ID der Eltern: [STEUER_ID_1].



In [10]:
# ── Inspect findings from guard.text ──────────────────────────────────────────
print(f"{'Type':<22} {'Original':<40} {'Score':>6}")
print("-" * 72)
for f in result_anon["findings"]:
    print(f"{f['type']:<22} {repr(f['original_text']):<40} {f['score']:>6.2f}")

Type                   Original                                  Score
------------------------------------------------------------------------
PERSON                 'Anna Richter'                             0.90
DATE                   '15. April 2008'                           0.85
LOCATION               'Musterstraße'                             0.90
ZIPCODE                '10115'                                    0.75
LOCATION               'Berlin'                                   0.90
PHONE_NUMBER           '+49 162 55512345'                         0.75
EMAIL_ADDRESS          'a.richter@gymnasium-berlin.de'            0.95
URL                    'https://lernportal.gymnasium-berlin.de/profil'   0.70
TIME                   '08:00 Uhr'                                0.70
IP_ADDRESS             '172.16.5.23'                              0.80
SVNR                   '12 150780 J 009'                          0.90
STEUER_ID              '12345678903'                              0.

## 5. Document Processing

`analyze.doc()` and `guard.doc()` accept `.txt`, `.pdf`, and `.docx` paths.  
The sample files live in `examples/files/`.

In [11]:
import os

# Resolve the files directory whether we run from repo root or examples/
FILES_DIR = os.path.join(os.path.dirname(os.path.abspath("")), "examples", "files")
if not os.path.isdir(FILES_DIR):
    FILES_DIR = os.path.join(os.getcwd(), "files")

TXT_FILE  = os.path.join(FILES_DIR, "praktikum_bewerbung.txt")
PDF_FILE  = os.path.join(FILES_DIR, "praktikum_bewerbung.pdf")
DOCX_FILE = os.path.join(FILES_DIR, "praktikum_bewerbung.docx")

print("Sample files:")
for p in [TXT_FILE, PDF_FILE, DOCX_FILE]:
    status = "✓" if os.path.exists(p) else "✗"
    print(f"  {status}  {p}")

Sample files:
  ✓  c:\Users\hibra\Desktop\github\deutsch-deid\examples\files\praktikum_bewerbung.txt
  ✓  c:\Users\hibra\Desktop\github\deutsch-deid\examples\files\praktikum_bewerbung.pdf
  ✓  c:\Users\hibra\Desktop\github\deutsch-deid\examples\files\praktikum_bewerbung.docx


In [19]:
from deutsch_deid.processors.doc_processor import read as doc_read


findings = analyze.doc(TXT_FILE)
print(f"── TXT — {len(findings)} entities ──────────────────────────────")
print(f"{'Type':<22} {'Score':>6}")
print("-" * 30)
for f in findings:
    print(f"{f['type']:<22} {f['score']:>6.2f}")

── TXT — 15 entities ──────────────────────────────
Type                    Score
------------------------------
PERSON                   0.90
LOCATION                 0.85
DATE                     0.85
DATE                     0.60
TIME                     0.70
PHONE_NUMBER             0.75
EMAIL_ADDRESS            0.95
URL                      0.85
LOCATION                 0.85
ZIPCODE                  0.55
SVNR                     0.90
STEUER_ID                0.92
IP_ADDRESS               0.80
LOCATION                 0.85
PERSON                   0.85


In [13]:
# ── Guard .pdf — tag mode ─────────────────────────────────────────────────────
pdf_result = guard.doc(PDF_FILE, config={"mode": "tag"})

print("Guarded PDF text")
print(pdf_result["guarded_text"])

Guarded PDF text
Betreff: Bewerbung um ein Praktikum im Bereich Data Science
Sehr geehrte Damen und Herren,
mein Name ist [PERSON] und ich bewerbe mich hiermit um ein
Praktikum in Ihrem Unternehmen in [LOCATION].
Ich wurde am [DATE] geboren und bin derzeit Student der Informatik.
Fuer ein persoenliches Gespraech stehe ich Ihnen gerne am [DATE]
um [TIME] zur Verfuegung.
Sie koennen mich jederzeit telefonisch unter [PHONE_NUMBER] oder per
E-Mail unter [EMAIL_ADDRESS] erreichen. Weitere Informationen
finden Sie auch auf meiner
Website: [URL]
Meine aktuelle Adresse lautet:
[PERSON] 12
[ZIPCODE] Berlin
Deutschland
Zusaetzlich moechte ich Ihnen einige relevante Identifikationsdaten
mitteilen, die fuer den Bewerbungsprozess erforderlich sein koennten:
Sozialversicherungsnummer: [SVNR]
Steuer-ID: [STEUER_ID]
Meine letzte Anmeldung im Bewerbungsportal erfolgte von der
IP-Adresse [IP_ADDRESS].
Ich freue mich sehr ueber die Moeglichkeit, praktische Erfahrungen in
Ihrem Unternehmen zu sammeln und 

In [16]:
# ── Guard .docx — anonymize mode ──────────────────────────────────────────────
docx_result = guard.doc(DOCX_FILE, config={"mode": "anonymize"})

print("Guarded DOCX text")
print(docx_result["guarded_text"])

Guarded DOCX text
Betreff: Bewerbung um ein Praktikum im Bereich Data Science

Sehr geehrte Damen und Herren,

mein Name ist Lukas Bauer und ich bewerbe mich hiermit um ein Praktikum in Ihrem Unternehmen in Hamburg.

Ich wurde am 27. April 1990 geboren und bin derzeit Student der Informatik. Für ein persönliches Gespräch stehe ich Ihnen gerne am 14. Januar 1983 um 09:15 Uhr zur Verfügung.

Sie können mich jederzeit telefonisch unter +49 151 87654321 oder per E-Mail unter l.bauer@beispiel.de erreichen. Weitere Informationen finden Sie auch auf meiner

Website: https://www.beispiel.de/seite
 

Meine aktuelle Adresse lautet:
Hauptstraße 12
20095 Berlin
Deutschland

Zusätzlich möchte ich Ihnen einige relevante Identifikationsdaten mitteilen, die für den Bewerbungsprozess erforderlich sein könnten:

Sozialversicherungsnummer: 65 230561 M 013
Steuer-ID: 87654321095

Meine letzte Anmeldung im Bewerbungsportal erfolgte von der IP-Adresse 10.20.30.40.

Ich freue mich sehr über die Möglichkeit, 

## 6. Custom Patterns

Use `custom_pattern()` to add your own regex-based entity types.

| Param | Description |
|-------|-------------|
| `name` | Entity label |
| `regex` | Python regex |
| `score` | Base confidence |
| `context` | Nearby words that boost score |
| `anonymize_list` | Fake-value pool for `anonymize` mode |

In [20]:
# ── Custom pattern: school student ID ─────────────────────────────────────────
student_id_pattern = custom_pattern(
    name="STUDENT_ID",
    regex=r"STU-\d{4}",
    score=0.90,
    context=["schüler", "student", "matrikelnummer", "schulnummer"],
    anonymize_list=["STU-9999", "STU-8888", "STU-7777"],
)

custom_text = "Schüler STU-1234 hat heute die Abschlussprüfung abgelegt."

custom_findings = analyze.text(
    custom_text,
    config={"custom_patterns": [student_id_pattern]},
)
print("Findings:", custom_findings)

Findings: [{'type': 'STUDENT_ID', 'start': 8, 'end': 16, 'score': 1.0}]


In [21]:
# ── Guard with custom pattern ──────────────────────────────────────────────────
for mode in ("anonymize", "tag", "i_tag"):
    result = guard.text(
        custom_text,
        config={"custom_patterns": [student_id_pattern], "mode": mode},
    )
    print(f"[{mode:10}] {result['guarded_text']}")

[anonymize ] Schüler STU-9999 hat heute die Abschlussprüfung abgelegt.
[tag       ] Schüler [STUDENT_ID] hat heute die Abschlussprüfung abgelegt.
[i_tag     ] Schüler [STUDENT_ID_1] hat heute die Abschlussprüfung abgelegt.


## 7. Config Options

### 7.1 Entity Filtering

In [22]:
# ── Keep only specific entity types ──────────────────────────────────────────
id_only = analyze.text(
    sample_text,
    config={"set_entities": {"keep": ["SVNR", "STEUER_ID", "PERSON"]}},
)
print("Keep [SVNR, STEUER_ID, PERSON]:")
for f in id_only:
    matched = sample_text[f['start']:f['end']].replace('\n', ' ')
    print(f"  {f['type']:<22} '{matched}'  score={f['score']:.2f}")

Keep [SVNR, STEUER_ID, PERSON]:
  PERSON                 'Anna Richter'  score=0.90
  SVNR                   '12 150780 J 009'  score=0.90
  STEUER_ID              '12345678903'  score=0.92


In [23]:
# ── Ignore specific entity types ──────────────────────────────────────────────
no_dates = analyze.text(
    sample_text,
    config={"set_entities": {"ignore": ["DATE", "TIME", "ZIPCODE"]}},
)
print("Ignore [DATE, TIME, ZIPCODE]:")
for f in no_dates:
    matched = sample_text[f['start']:f['end']].replace('\n', ' ')
    print(f"  {f['type']:<22} '{matched}'")

Ignore [DATE, TIME, ZIPCODE]:
  PERSON                 'Anna Richter'
  LOCATION               'Musterstraße'
  LOCATION               'Berlin'
  PHONE_NUMBER           '+49 162 55512345'
  EMAIL_ADDRESS          'a.richter@gymnasium-berlin.de'
  URL                    'https://lernportal.gymnasium-berlin.de/profil'
  IP_ADDRESS             '172.16.5.23'
  SVNR                   '12 150780 J 009'
  STEUER_ID              '12345678903'


### 7.2 Score Threshold

In [24]:
# ── Compare different thresholds ──────────────────────────────────────────────
for threshold in (0.0, 0.5, 0.7, 0.9):
    results = analyze.text(sample_text, config={"score_threshold": threshold})
    types = ", ".join(sorted({r['type'] for r in results}))
    print(f"threshold={threshold:.1f}  →  {len(results):2d} findings  [{types}]")

threshold=0.0  →  12 findings  [DATE, EMAIL_ADDRESS, IP_ADDRESS, LOCATION, PERSON, PHONE_NUMBER, STEUER_ID, SVNR, TIME, URL, ZIPCODE]
threshold=0.5  →  12 findings  [DATE, EMAIL_ADDRESS, IP_ADDRESS, LOCATION, PERSON, PHONE_NUMBER, STEUER_ID, SVNR, TIME, URL, ZIPCODE]
threshold=0.7  →  12 findings  [DATE, EMAIL_ADDRESS, IP_ADDRESS, LOCATION, PERSON, PHONE_NUMBER, STEUER_ID, SVNR, TIME, URL, ZIPCODE]
threshold=0.9  →   6 findings  [EMAIL_ADDRESS, LOCATION, PERSON, STEUER_ID, SVNR]


### 7.3 Scoring Deep-Dive

The same entity type can receive very different scores depending on context and validation.

In [25]:
scoring_cases = [
    # (description,                         text,                                  entity)
    ("Phone — no context",                  "+49 162 55512345",                     "PHONE_NUMBER"),
    ("Phone — with 'Telefon' context",      "Telefon: +49 162 55512345",            "PHONE_NUMBER"),
    ("SVNR — regex only (no context)",      "12 150780 J 009",                      "SVNR"),
    ("SVNR — with context + valid",         "Sozialversicherungsnummer: 12 150780 J 009", "SVNR"),
    ("USt-IdNr — valid",                    "USt-IdNr: DE123456788",                "STEUER_ID"),
    ("USt-IdNr — invalid check digit",      "USt-IdNr: DE123456789",                "STEUER_ID"),
    ("Individual TIN — valid (no context)", "12345678903",                          "STEUER_ID"),
]

print(f"{'Description':<40} {'Score':>6}")
print("-" * 48)
for desc, text, entity in scoring_cases:
    res = analyze.text(text, config={"set_entities": {"keep": [entity]}})
    score = f"{res[0]['score']:.2f}" if res else "  n/a"
    print(f"{desc:<40} {score:>6}")

Description                               Score
------------------------------------------------
Phone — no context                         0.70
Phone — with 'Telefon' context             0.75
SVNR — regex only (no context)             0.90
SVNR — with context + valid                0.90
USt-IdNr — valid                           0.92
USt-IdNr — invalid check digit              n/a
Individual TIN — valid (no context)        0.55


## 8. End-to-End — German Student Record

A realistic school-administration record containing all 11 entity types, processed through all three guard modes.

In [26]:
student_record = """Schülerakte — Gymnasium Muster Berlin

Persönliche Daten:
  Name:               Tobias Koch
  Geburtsdatum:       03. September 2007
  Adresse:            Berliner Str. 21, 10115 Berlin
  Telefon (Eltern):   +49 30 98765432
  E-Mail (Schüler):   t.koch@gymnasium-muster.de
  Lernportal:         https://lernportal.gymnasium-muster.de/schueler/tkoch
  Unterrichtsbeginn:  07:50 Uhr
  Schulnetz-IP:       10.0.2.15

Offizielle Identifikation:
  Sozialversicherungsnummer:  65 230561 M 013
  Steuer-ID (Erziehungsber.): 87654321095
  USt-IdNr (Schulträger):     DE876543219
"""

print(student_record)

Schülerakte — Gymnasium Muster Berlin

Persönliche Daten:
  Name:               Tobias Koch
  Geburtsdatum:       03. September 2007
  Adresse:            Berliner Str. 21, 10115 Berlin
  Telefon (Eltern):   +49 30 98765432
  E-Mail (Schüler):   t.koch@gymnasium-muster.de
  Lernportal:         https://lernportal.gymnasium-muster.de/schueler/tkoch
  Unterrichtsbeginn:  07:50 Uhr
  Schulnetz-IP:       10.0.2.15

Offizielle Identifikation:
  Sozialversicherungsnummer:  65 230561 M 013
  Steuer-ID (Erziehungsber.): 87654321095
  USt-IdNr (Schulträger):     DE876543219



In [27]:
# ── Analyze: show all detected PII ────────────────────────────────────────────
record_findings = analyze.text(student_record)

print(f"Detected {len(record_findings)} PII entities:\n")
print(f"{'Type':<22} {'Matched text':<45} {'Score':>6}")
print("-" * 77)
for f in sorted(record_findings, key=lambda x: x['start']):
    matched = student_record[f['start']:f['end']].replace('\n', ' ')
    print(f"{f['type']:<22} {matched:<45} {f['score']:>6.2f}")

Detected 13 PII entities:

Type                   Matched text                                   Score
-----------------------------------------------------------------------------
LOCATION               Berlin                                          0.85
PERSON                 Tobias Koch                                     0.90
DATE                   03. September 2007                              0.85
ZIPCODE                10115                                           0.55
LOCATION               Berlin                                          0.85
PHONE_NUMBER           +49 30 98765432                                 0.75
EMAIL_ADDRESS          t.koch@gymnasium-muster.de                      0.95
URL                    https://lernportal.gymnasium-muster.de/schueler/tkoch   0.70
TIME                   07:50 Uhr                                       0.70
IP_ADDRESS             10.0.2.15                                       0.80
SVNR                   65 230561 M 013             

In [28]:
# ── Guard: tag mode — bracket labels ──────────────────────────────────────────
tagged = guard.text(student_record, config={"mode": "tag"})
print("=== TAG MODE ===")
print(tagged["guarded_text"])

=== TAG MODE ===
Schülerakte — Gymnasium Muster [LOCATION]

Persönliche Daten:
  Name:               [PERSON]
  Geburtsdatum:       [DATE]
  Adresse:            Berliner Str. 21, [ZIPCODE] [LOCATION]
  Telefon (Eltern):   [PHONE_NUMBER]
  E-Mail (Schüler):   [EMAIL_ADDRESS]
  Lernportal:         [URL]
  Unterrichtsbeginn:  [TIME]
  Schulnetz-IP:       [IP_ADDRESS]

Offizielle Identifikation:
  Sozialversicherungsnummer:  [SVNR]
  Steuer-ID (Erziehungsber.): [STEUER_ID]
  USt-IdNr (Schulträger):     [STEUER_ID]



In [29]:
# ── Guard: anonymize — realistic German fakes ──────────────────────────────────
anonymized = guard.text(student_record, config={"mode": "anonymize"})
print("=== ANONYMIZED ===")
print(anonymized["guarded_text"])

=== ANONYMIZED ===
Schülerakte — Gymnasium Muster Hamburg

Persönliche Daten:
  Name:               Lukas Bauer
  Geburtsdatum:       14. Januar 1983
  Adresse:            Berliner Str. 21, 20095 Hamburg
  Telefon (Eltern):   +49 151 87654321
  E-Mail (Schüler):   l.bauer@beispiel.de
  Lernportal:         https://www.beispiel.de/seite
  Unterrichtsbeginn:  09:15 Uhr
  Schulnetz-IP:       10.20.30.40

Offizielle Identifikation:
  Sozialversicherungsnummer:  12 150780 J 009
  Steuer-ID (Erziehungsber.): 12345678903
  USt-IdNr (Schulträger):     DE123456788



In [30]:
# ── Guard: i_tag — indexed labels (same entity = same index) ──────────────────
indexed = guard.text(student_record, config={"mode": "i_tag"})
print("=== INDEXED TAG MODE ===")
print(indexed["guarded_text"])

=== INDEXED TAG MODE ===
Schülerakte — Gymnasium Muster [LOCATION_1]

Persönliche Daten:
  Name:               [PERSON_1]
  Geburtsdatum:       [DATE_1]
  Adresse:            Berliner Str. 21, [ZIPCODE_1] [LOCATION_1]
  Telefon (Eltern):   [PHONE_NUMBER_1]
  E-Mail (Schüler):   [EMAIL_ADDRESS_1]
  Lernportal:         [URL_1]
  Unterrichtsbeginn:  [TIME_1]
  Schulnetz-IP:       [IP_ADDRESS_1]

Offizielle Identifikation:
  Sozialversicherungsnummer:  [SVNR_1]
  Steuer-ID (Erziehungsber.): [STEUER_ID_1]
  USt-IdNr (Schulträger):     [STEUER_ID_2]



In [31]:
# ── Guard: high-confidence only (threshold=0.7) ───────────────────────────────
strict = guard.text(student_record, config={"mode": "tag", "score_threshold": 0.7})
print("=== TAG (score_threshold=0.7) ===")
print(strict["guarded_text"])
print(f"\n{len(strict['findings'])} findings retained")

=== TAG (score_threshold=0.7) ===
Schülerakte — Gymnasium Muster [LOCATION]

Persönliche Daten:
  Name:               [PERSON]
  Geburtsdatum:       [DATE]
  Adresse:            Berliner Str. 21, 10115 [LOCATION]
  Telefon (Eltern):   [PHONE_NUMBER]
  E-Mail (Schüler):   [EMAIL_ADDRESS]
  Lernportal:         [URL]
  Unterrichtsbeginn:  [TIME]
  Schulnetz-IP:       [IP_ADDRESS]

Offizielle Identifikation:
  Sozialversicherungsnummer:  [SVNR]
  Steuer-ID (Erziehungsber.): [STEUER_ID]
  USt-IdNr (Schulträger):     [STEUER_ID]


12 findings retained
